# Reproduce Binding Site Prediction on Temporal Set
Here we provide a reproducibility script to predict binding sites on the temporal set using the SMARTBind model. The script loads the pre-trained SMARTBind models, predict binding sites for a given RNA-ligand pair, and averages the predictions across multiple models, and finally calculate AUC scores with respect to ground truth binding sites.

In [ ]:
import warnings

import pandas as pd
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")

import sys
from pathlib import Path


def _repo_root():
    """Walk up to the checkout that contains the smartbind package."""
    for folder in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (folder / "smartbind" / "__init__.py").is_file():
            return folder
    raise SystemExit("Run this notebook from inside a checkout of the SMARTBind repository.")


sys.path.insert(0, str(_repo_root()))
import pickle
import logging

from smartbind.preprocess import convert_smiles_to_pf2
from smartbind import BindingPL
import os

for logger_name in [
    "lightning_fabric.utilities.seed",
    "pytorch_lightning.utilities.seed",
    "lightning.pytorch.utilities.seed",
]:
    logging.getLogger(logger_name).setLevel(logging.WARNING)

In [ ]:
def predict_binding(target_rna_chain, ligand_fp2, best_model_paths):
    binding_site_predictions_dict = {i: [] for i in range(len(target_rna_chain))}
    for weight_name in sorted(w for w in os.listdir(best_model_paths) if w.endswith('.pth')):
        weight_path = os.path.join(best_model_paths, weight_name)
        smartbind_binding_model = BindingPL(device='cpu', vs_mode=True)
        smartbind_binding_model.load_pretrained_model(model_path=weight_path, device='cpu', mode='inference')
        binding_prediction = smartbind_binding_model.predict_binding(target_rna_chain, ligand_fp2)
        binding_prediction = binding_prediction.detach().numpy()
        normalized_binding_prediction = (binding_prediction - min(binding_prediction)) / (max(binding_prediction) - min(binding_prediction)) 
        for i in range(len(target_rna_chain)):
            binding_site_predictions_dict[i].append(normalized_binding_prediction[i])

    average_predictions = []
    for i in range(len(target_rna_chain)):
        average_predictions.append(sum(binding_site_predictions_dict[i]) / 10)

    normalized_binding_site_prediction = ((average_predictions - min(average_predictions)) /
                                        (max(average_predictions) - min(average_predictions)))
    normalized_binding_site_prediction = [round(i, 3) for i in normalized_binding_site_prediction]
    normalized_binding_site_prediction = [float(i) for i in normalized_binding_site_prediction]
    return normalized_binding_site_prediction

In [ ]:
result_list = {}
raw_result = {}
gt_pd = pd.read_csv('temporal_test_binding_groundtruth.csv')
for i in range(len(gt_pd)):
    pdbId = gt_pd.loc[i, 'PDB ID']
    print(f"Processing {i+1}/{len(gt_pd)}: PDB ID {pdbId}")
    ligandId = gt_pd.loc[i, 'Ligand ID']
    rna_sequence = gt_pd.loc[i, 'RNA sequence']
    ligand_smiles = gt_pd.loc[i, 'SMILES']
    binding_gt = gt_pd.loc[i, 'binding_groundtruth']
    normalized_predict = predict_binding(rna_sequence, convert_smiles_to_pf2(ligand_smiles), '../../SMARTBind_weight')
    normalized_predict = [round(i, 3) for i in normalized_predict]
    gt_list = [int(i) for i in binding_gt]
    raw_result[f'{pdbId}_{ligandId}'] = {
        'groundtruth': gt_list,
        'prediction': normalized_predict
    }
    auc_score = roc_auc_score(gt_list, normalized_predict)
    print(f"PDB ID: {pdbId}, Ligand ID: {ligandId}, AUC Score: {auc_score:.4f}")
    result_list[f'{pdbId}_{ligandId}'] = auc_score
# save raw results
with open('binding_site_predictions.pkl', 'wb') as f:
    pickle.dump(raw_result, f)

In [ ]:
result_list

In [ ]:
# Calculate mean
mean_auc = sum(result_list.values()) / len(result_list)
print(f"Mean AUC Score across all RNA-ligand pairs: {mean_auc:.4f}")

In [ ]:
# Mean AUC Score across all RNA-ligand pairs: 0.7451